# Caso 5: localización de depósitos con múltiples objetivos

---


## Instrucciones generales

El primer paso antes de resolver este laboratorio es leer y entender el **enunciado del caso**.

Este laboratorio tiene las siguientes secciones:
* **Formulación**: en este caso particular, definimos dos funciones objetivo $z_1$ y $z_2$
* **Importación de librerías**
* **Creación de parámetros**
* **Modelado**: en esta práctica, haremos tres (3) implementaciones del mismo problema:
    * **Minimización de costos**
    * **Maximización de la satisfacción**
    * **Maximización de la satisfacción con restricción de costos**
* **Reporte de Resultados**

Este tipo de actividades se evaluará sobre un total de 100 puntos. Las celdas calificables se distinguen por tener la instrucción `# your code here`. Antes de estas celdas encontrarás instrucciones y consejos para resolver las preguntas, también el puntaje que le corresponde.

¡Éxitos!

## Formulación
---

Te presentamos la formulación del caso de la semana de forma resumida. Te recomendamos revisar la formulación una vez hayas leído el enunciado del caso. Es bueno que te familiarices con los elementos de la formulación antes de iniciar la implementación.

### Conjuntos y Parámetros
>#### **Conjuntos**
>* $I:$ Depósitos
>* $J:$ Centros de acopio consolidados (CACs)

>#### **Parámetros**
>* $k_i:$ Capacidad del depósito $i \in I$ (miles de toneladas por año)
>* $f_i:$ Costo de operación del depósito $i \in I$ (millones de pesos por año)
>* $d_j:$ Producción proyectada del CAC $j\in J$ (miles de tonaledas por año)
>* $q:$ Costo anualizado (por cada mil toneladas por kilómetro) de transportar café
>* $r:$ Distancia máxima (en kilómetros) entre el CAC y el depósito asignado para estar "bien" atendido
>* $h_{ij}:$ Distancia (en kilómetros) entre el CAC $j \in J$ y el depósito $i \in I$
>* $c_{ij}:$ Costo anualizado de atender el CAC $j\in J$ con el depósito $i\in I$ (Se calcula como: $c_{ij} = q\cdot d_j \cdot h_{ij}$)

### Variables de Decisión
>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$
    
### Restricciones
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

> **Naturaleza de variables**
>>$x_{ij} \in \{0,1\} , \;\forall i \in I, j \in J$
>>
>>$y_{i} \in \{0,1\} , \;\forall i \in I$

### Función Objetivo
>* Minimizar los costos totales de operación y transporte
>>`# Para desarrollo del estudiante`
>* Maximizar satisfacción de los CACs
>>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

## Importación de librerías
---
En esta práctica usaremos:
* El paquete `pandas` es muy útil para el análisis de datos en general. Le asignamos el alias de `pd`.
* El paquete `pulp` permite crear modelos de optimización, crear variables, añadir restricciones y muchos más. Le asignamos el alias de `lp`.
* La función `distance` del módulo `geopy.distance` nos permite hallar fácilmente la distancia geodéisca en kilómetros entre dos pares de coordenadas de longitud y latitud.


In [65]:
import pandas as pd
import pulp as lp
from geopy.distance import distance

### Librerías: qué hace cada `import`

**Línea 1 — `import pandas as pd`**
- **Pandas** sirve para manejar **tablas** (como el Excel) con nombre `pd`.
- Aquí lo usan para leer hojas y armar `DataFrame`.

**Línea 2 — `import pulp as lp`**
- **PuLP** construye modelos de optimización (variables, sumas, restricciones).
- El alias `lp` evita escribir `pulp.` cada vez.

**Línea 3 — `from geopy.distance import distance`**
- **Geopy** calcula distancias entre **pares de coordenadas** (latitud, longitud).
- `distance(puntoA, puntoB)` devuelve un objeto; en el notebook usan `.kilometers` para obtener km.

**En una frase:** Pandas trae datos; PuLP modelo y resuelve; Geopy mide qué tan lejos queda cada CAC de cada depósito.

---



## Creación de Parámetros
---

### Lectura del archivo de soporte

Los datos que necesitamos para esta práctica se encuentran disponibles en el archivo `Soporte Caso 5.xlsx`.
En este archivo encontraremos los mismo datos del enunciado.
Importamos las hojas `CACs` y `Depositos` del archivo `Soporte Caso 5.xlsx`.
Estas hojas son importadas como objetos `DataFrame` de `pandas`.

In [ ]:
# Leer datos del Excel: hojas 'CACs' y 'Depositos'
# cacs: DataFrame con datos de Centros de Acopio (municipio, producción, lat/lon)
# depositos: DataFrame con datos de depósitos candidatos (municipio, capacidad, costo fijo, lat/lon)
cacs = pd.read_excel('Soporte Caso 5.xlsx', sheet_name='CACs')
depositos = pd.read_excel('Soporte Caso 5.xlsx', sheet_name='Depositos')

### Lectura del Excel (hojas `CACs` y `Depositos`)

**Primera línea — `cacs = pd.read_excel(..., sheet_name='CACs')`**
- Abre el archivo `Soporte Caso 5.xlsx` y guarda la hoja de **CACs** en la variable `cacs`.
- Cada **fila** es un centro de acopio; columnas típicas: municipio, producción, latitud, longitud.

**Segunda línea — `depositos = pd.read_excel(..., sheet_name='Depositos')`**
- Igual, pero la hoja de **depósitos candidatos**.
- Ahí vienen capacidad, costo fijo y coordenadas por municipio.

**Qué esperas después:** dos tablas en memoria listas para convertirlas en listas y diccionarios del modelo.

**CAC** = *Centro de Acopio Consolidado* (origen del café en el caso).

---



### Procesamiento de archivos de soporte

En este paso, se crean los **Conjuntos** y **Parámetros**.
Es necesario dejar todo expresado en términos de listas y diccionarios para facilitar la implementación del modelo en PuLP. Adicionalmente, debemos procesar las coordenadas de longitud y latitud para obtener las distancias entre CACs y Depósitos.

In [ ]:
# Crear conjuntos y parámetros
# I: lista de municipios con depósitos candidatos
I = depositos.Municipio.to_list()
# J: lista de municipios con CACs
J = cacs.Municipio.to_list()

# Diccionarios de parámetros
# capacidad[i]: capacidad del depósito i (miles de toneladas/año)
capacidad = {row["Municipio"]: row["Capacidad"] for _, row in depositos.iterrows()}
# costo_fijo[i]: costo de operación del depósito i (millones de pesos/año)
costo_fijo = {row["Municipio"]: row["CostoFijo"] for _, row in depositos.iterrows()}
# depositos_lat_lon[i]: tupla (latitud, longitud) del depósito i
depositos_lat_lon = {
    row["Municipio"]: (row["Latitud"], row["Longitud"])
    for _, row in depositos.iterrows()
}

# produccion[j]: producción del CAC j (miles de toneladas/año)
produccion = {row["Municipio"]: row["Produccion"] for _, row in cacs.iterrows()}
# cacs_lat_lon[j]: tupla (latitud, longitud) del CAC j
cacs_lat_lon = {
    row["Municipio"]: (row["Latitud"], row["Longitud"]) for _, row in cacs.iterrows()
}

# Parámetros fijos
q = 90  # Costo por mil toneladas por km (pesos anualizados)
r = 125  # Distancia máxima para "satisfacción" (km)

# Calcular distancias entre cada depósito i y CAC j usando geopy
distancia = {
    (i, j): distance(depositos_lat_lon[i], cacs_lat_lon[j]).kilometers
    for i in I
    for j in J
}

### Esta celda: conjuntos y diccionarios (línea por línea)

**`I = depositos.Municipio.to_list()`**
- `I` es la lista de **nombres de municipio** donde puede haber depósito. En fórmulas del curso, el conjunto de depósitos.

**`J = cacs.Municipio.to_list()`**
- `J` es la lista de municipios donde está cada **CAC**. En fórmulas, el conjunto de CACs.

**`capacidad = { row['Municipio']: row['Capacidad'] ... }`**
- **Diccionario:** clave = nombre del depósito, valor = capacidad (máximo que puede recibir al año).
- Así escribes `capacidad[i]` cuando `i` es un municipio.

**`costo_fijo = { ... }`**
- Igual idea: municipio → costo fijo anual de operar ese depósito.

**`depositos_lat_lon` y `cacs_lat_lon`**
- Cada uno mapea municipio → `(latitud, longitud)` para calcular distancias.

**`q = 90` y `r = 125`**
- `q`: factor económico del enunciado (costo por mil toneladas–km).
- `r`: radio en **km** para decir si un CAC queda “bien atendido” en el indicador `z2`.

**`distancia = { (i,j): ... for i in I for j in J }`**
- Doble `for`: recorre **cada** depósito y **cada** CAC.
- Para cada par guarda km entre coordenadas (`distance(...).kilometers`).
- La llave es `(i, j)` porque más adelante se usa `distancia[(i, j)]`.

---



In [68]:
distancia

{('Medellín, Antioquia', 'Andes, Antioquia'): 81.6407744153149,
 ('Medellín, Antioquia', 'Medellín, Antioquia'): 0.0,
 ('Medellín, Antioquia', 'Dabeiba, Antioquia'): 107.79278431038877,
 ('Medellín, Antioquia', 'Salgar, Antioquia'): 53.335670657757895,
 ('Medellín, Antioquia', 'San Pablo de Borbur, Boyacá'): 177.7022202999153,
 ('Medellín, Antioquia', 'Labranzagrande, Boyacá'): 341.843300383342,
 ('Medellín, Antioquia', 'Miraflores, Boyacá'): 295.50792681396376,
 ('Medellín, Antioquia', 'Moniquirá, Boyacá'): 230.85325502996707,
 ('Medellín, Antioquia', 'Manizales, Caldas'): 132.50832910815635,
 ('Medellín, Antioquia', 'Anserma, Caldas'): 119.36408078562327,
 ('Medellín, Antioquia', 'Pensilvania, Caldas'): 106.38936050657013,
 ('Medellín, Antioquia', 'Riosucio, Caldas'): 91.7929643629694,
 ('Medellín, Antioquia', 'Aguadas, Caldas'): 77.80655349448206,
 ('Medellín, Antioquia', 'Morales, Cauca'): 399.20320142759863,
 ('Medellín, Antioquia', 'El Tambo, Cauca'): 442.9927758625815,
 ('Medell

## Guía rápida: siglas y qué es cada cosa en el código

Lee esto antes de las preguntas. Si algo suena técnico, aquí va en palabras simples.

### Siglas
- **CAC** — *Centro de Acopio Consolidado*. Es cada lugar donde se junta el café antes de enviarlo al depósito. En el Excel, cada fila de la hoja `CACs` es un CAC (identificado por municipio).
- **MILP / modelo entero** — Problema de optimización donde algunas decisiones son **sí o no** (0 o 1). Aquí: abrir depósito y asignar CAC a depósito.
- **PuLP (`lp`)** — Librería de Python que **escribe** el modelo matemático en forma que un **solver** (por detrás, CBC) lo resuelva.
- **FO** — Función objetivo: lo que el programa **minimiza** o **maximiza** (costo o satisfacción).

### Nombres que verás una y otra vez
| Símbolo / nombre | Qué representa en la vida real |
|------------------|--------------------------------|
| `I` | Lista de **depósitos candidatos** (nombres de municipio del Excel de depósitos). |
| `J` | Lista de **CACs** (nombres de municipio del Excel de CACs). |
| `i` | Un depósito concreto cuando haces `for i in I`. |
| `j` | Un CAC concreto cuando haces `for j in J`. |
| `capacidad[i]` | Cuánto puede recibir al año el depósito `i` (miles de toneladas; en el diccionario está por municipio). |
| `costo_fijo[i]` | Cuánto cuesta **mantener abierto** un año el depósito `i` (millones de pesos en el Excel). |
| `produccion[j]` | Cuánto café produce al año el CAC `j` (miles de toneladas). |
| `distancia[(i, j)]` | Kilómetros entre el depósito `i` y el CAC `j` (salen de latitud/longitud con `geopy`). |
| `q` | Parámetro del caso: “cuánto cuesta” transportar; en el código está fijo en **90** (pesos anualizados por mil ton–km). |
| `r` | **Radio en km** (125 en el código). Si la asignación queda **más lejos que r**, para la **satisfacción** `z2` **no cuenta** como servicio cercano (pero el viaje igual puede existir si minimiza costo). |
| `x[i, j]` | Variable **decisión**: 1 si el CAC `j` es atendido por el depósito `i`; 0 si no. |
| `y[i]` | Variable **decisión**: 1 si abrimos el depósito `i`; 0 si no. |
| `z1` | **Costo total** (fijos + transporte). A veces es el **objetivo** (minimizar), a veces solo se **calcula** después con `lp.value`. |
| `z2` | **Satisfacción**: suma de producción de CACs que queden asignados a un depósito **a distancia ≤ r**. Mide “cuánta tonelada queda bien atendida”. |
| `problema` | Objeto PuLP donde sumas la FO (`+=`) y las restricciones. |
| `R1`, `R2`, `R3` | Nombres de **restricciones**. R1 = un depósito por CAC; R2 = capacidad y coherencia con apertura; R3 = tope de costo en el tercer modelo. |

### Código que ya viene en el notebook (no lo escribes tú, pero lo usas)
- **`pd.read_excel(...)`** — Carga el Excel en una **tabla** (`cacs`, `depositos`). Es como abrir la hoja en Python.
- **`{ clave: valor for ... }`** — **Diccionario por comprensión**: recorres filas o listas y llenas “llave → valor” de un solo golpe.
- **`lp.LpVariable.dicts(...)`** — Crea muchas variables `x` e `y` indexadas por `(i,j)` o por `i`.
- **`lp.lpSum(expresion for ...)`** — Escribe una **suma** para PuLP; **no** es un `for` que imprima números, es la forma algebraica que el solver entiende.
- **`problema.solve()`** — Manda a resolver; después puedes leer valores con `.value()` o `lp.value(...)`.

---




**Pregunta 1 (10 puntos)**

* Crea el parámetro de costo de transporte $c_{ij}$ en un diccionario llamado `costo_transporte`
* Las **llaves** de este diccionario deben ser los pares $(i,j)$, es decir, (depósitos, CACs)
* Los **valores** de este diccionario deben ser los costos de transporte definidos en la formulación

### Explicación detallada — Pregunta 1 (`costo_transporte`)

**Qué te piden:** armar el costo de transporte **por ruta** entre cada depósito y cada CAC.

**Siglas:** **CAC** = Centro de Acopio Consolidado. **i** = depósito (recorre `I`). **j** = CAC (recorre `J`).

**Fórmula (en papel):**  
\(c_{ij} = q \times d_j \times h_{ij}\)  
- \(d_j\) en código es `produccion[j]` (tonelaje del CAC).  
- \(h_{ij}\) en código es `distancia[(i, j)]` (km).  
- \(q\) es el factor económico fijado arriba en el notebook.

**El código, parte por parte:**
```text
costo_transporte = {
    (i, j): q * produccion[j] * distancia[(i, j)]
    for i in I
    for j in J
}
```
1. **`for i in I`** — “Para cada depósito de la lista…”.  
2. **`for j in J`** — “…y para cada CAC de la lista…”. Así cubres **todas** las rutas posibles.  
3. **`(i, j)`** — Es la **llave** del diccionario: el par depósito–CAC. Más abajo el modelo usa exactamente `costo_transporte[i, j]`.  
4. **`q * produccion[j] * distancia[(i, j)]`** — El valor guardado: costo anualizado de esa ruta si se usara a pleno.

**Qué esperas al ejecutar:** un diccionario con tantas entradas como \(|I| \times |J|\). Si falta algún par, más adelante el modelo fallaría al armar una suma.

**Por qué importa:** sin `costo_transporte` no puedes calcular la parte de **transporte** dentro del costo total `z1`.

---



In [ ]:
# Crear diccionario de costo de transporte c_ij = q * d_j * h_ij
# Llaves: (i, j) donde i es depósito, j es CAC
# Valores: costo anualizado de transportar producción de j a i
costo_transporte = {(i, j): q * produccion[j] * distancia[(i, j)] for i in I for j in J}

In [ ]:
# Forma alternativa 1: calcular costo_transporte en varias líneas con bucles explícitos
costo_transporte_alt1 = {}
for i in I:
    for j in J:
        costo = q * produccion[j] * distancia[(i, j)]
        costo_transporte_alt1[(i, j)] = costo

# Forma alternativa 2: usando comprensión de lista y luego dict()
pares = [(i, j) for i in I for j in J]
costo_transporte_alt2 = {par: q * produccion[par[1]] * distancia[par] for par in pares}

# Verificar que sean iguales
print("Alt1 == original:", costo_transporte_alt1 == costo_transporte)
print("Alt2 == original:", costo_transporte_alt2 == costo_transporte)

In [ ]:
# Forma alternativa de pruebas: imprimir más detalles
print(f"Número de depósitos (I): {len(I)}")
print(f"Número de CACs (J): {len(J)}")
print(f"Total pares posibles: {len(I) * len(J)}")
print("Primeros 5 pares de costo_transporte:")
for idx, (par, costo) in enumerate(costo_transporte.items()):
    if idx >= 5:
        break
    print(f"  {par}: ${costo:.2f}")

# Verificar rangos razonables
costos_lista = list(costo_transporte.values())
print(f"Costo mínimo: ${min(costos_lista):.2f}")
print(f"Costo máximo: ${max(costos_lista):.2f}")
print(f"Costo promedio: ${sum(costos_lista)/len(costos_lista):.2f}")

**Celda de Prueba (0 puntos)**

Es una buena práctica imprimir algunos objetos que contienen los parámetros en la consola luego de crearlos. De esta forma puedes corregir errores y familiarizarte con las estructuras de datos que se van a utilizar. Puedes hacer estas pruebas en la celda a continuación.

* **Esta celda no es calificable**

### Explicación — Celda de prueba (no califica)

**Qué hace:** solo **mira** si el diccionario anterior quedó bien.

**Línea por línea:**
- **`len(costo_transporte)`** — Cuenta cuántas rutas (pares) tienes. Debe coincidir con (número de depósitos) × (número de CACs).  
- **`list(costo_transporte.items())[:3]`** — `items()` devuelve pares (llave, valor). `[:3]` toma **solo tres** para no llenar la pantalla. Es una **muestra**.

**Qué esperas:** números razonables (no ceros everywhere, no errores). Si algo falló en Excel o en `distancia`, aquí lo notas antes del modelo grande.

---



In [ ]:
# Pruebas: verificar que los diccionarios se crearon correctamente
# len(costo_transporte): debe ser |I| * |J| (número total de pares)
print('Cantidad de pares (i,j):', len(costo_transporte))
# list(costo_transporte.items())[:3]: muestra primeros 3 pares (llave, valor) del diccionario
print('Muestra:', list(costo_transporte.items())[:3])

Cantidad de pares (i,j): 1045
Muestra: [(('Medellín, Antioquia', 'Andes, Antioquia'), 259887.0771962719), (('Medellín, Antioquia', 'Medellín, Antioquia'), 0.0), (('Medellín, Antioquia', 'Dabeiba, Antioquia'), 264652.8440388665)]


## Modelado - Minimización de costos ($z_1$)
---

### Declaración del modelo

In [ ]:
# Crear el problema de optimización con PuLP
# sense=lp.LpMinimize: minimizar la función objetivo (costos)
problema = lp.LpProblem(sense=lp.LpMinimize)

### Crear `problema` en modo **minimizar**

**Código:** `problema = lp.LpProblem(sense=lp.LpMinimize)`

**Qué es `problema`:** un objeto “vacío” donde después sumas:
1. La **función objetivo** (en la siguiente sección: minimizar costo `z1`).
2. Las **restricciones** R1, R2, …

**`sense=lp.LpMinimize`:** le dice a PuLP que la solución debe tener el **menor** valor posible del objetivo (en el primer bloque: menor costo).

**En palabras:** abres el formulario del primer escenario: “quiero gastar lo menos posible respetando reglas”.

---



### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [ ]:
# Variables de decisión
# x[i,j]: 1 si CAC j es atendido por depósito i, 0 otherwise (binaria)
x = lp.LpVariable.dicts("atender", [(i, j) for i in I for j in J], lowBound = 0, cat=lp.LpBinary)
# y[i]: 1 si se decide operar depósito i, 0 otherwise (binaria)
y = lp.LpVariable.dicts("operar", I, lowBound = 0, cat=lp.LpBinary)

### Qué son exactamente `x` e `y` (primer modelo)

**Línea 1 — `x = lp.LpVariable.dicts(...)`**
- Crea una **variable de decisión** por cada par (depósito `i`, CAC `j`).
- El texto `"atender"` es solo un **nombre** para el reporte del modelo.
- `[(i, j) for i in I for j in J]` recorre **todos** los depósitos y, para cada uno, **todos** los CACs: es la lista de “casillas” donde puede haber un sí/no de asignación.
- `lowBound = 0` y `cat=lp.LpBinary`: cada variable solo puede valer **0 o 1** (no asigno / sí asigno).

**Línea 2 — `y = lp.LpVariable.dicts(...)`**
- Una variable por cada depósito `i`.
- `LpBinary`: 1 = **abro** ese depósito; 0 = **no** lo uso.

**Qué espera el programa al resolver:** valores 0 o 1 que cumplan las reglas (cada CAC con un solo depósito, capacidad, etc.) y que hagan el **menor** costo total. Tú no eliges los números a mano; `solve()` los calcula.

---



### Función Objetivo
Minimizar los costos totales de operación y transporte
>`# Para desarrollo del estudiante`

**Pregunta 2 (10 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Explicación detallada — Pregunta 2 (minimizar costo `z1`)

**Contexto:** ya existe `problema = lp.LpProblem(sense=lp.LpMinimize)`. Esta celda le dice **qué** minimizar.

**En palabras sencillas:** “Suma todo lo que pagas en **fijos** por depósitos abiertos y todo lo que pagas en **transporte** por las asignaciones que elija el programa.”

**Fórmula:**  
\(\min z_1 = \sum_i f_i y_i + \sum_{i,j} c_{ij} x_{ij}\)

**El código:**
```text
problema += lp.lpSum(costo_fijo[i] * y[i] for i in I) + lp.lpSum(
    costo_transporte[i, j] * x[i, j] for i in I for j in J
)
```
1. **`problema += ...`** — En PuLP, esto **define la función objetivo** (la primera vez que sumas así al problema).  
2. **`lp.lpSum(costo_fijo[i] * y[i] for i in I)`** — Para cada depósito `i`: si `y[i]=1` sumas el costo fijo; si `y[i]=0` sumas 0.  
3. **`lp.lpSum(costo_transporte[i,j] * x[i,j] ...)`** — Para cada ruta: si `x[i,j]=1` sumas ese costo de transporte; si es 0, sumas 0.  
4. **`for i in I` / `for j in J`** — Recorren todos los índices para armar la suma completa.

**Tecnicismo útil:** `lpSum` no es un bucle Python que ya te dé el resultado final; construye una **expresión simbólica** que el solver optimiza.

---



In [ ]:
# Función objetivo: minimizar costos totales z1
# Costos fijos: suma de costo_fijo[i] * y[i] para depósitos abiertos
# Costos transporte: suma de costo_transporte[i,j] * x[i,j] para asignaciones
problema += lp.lpSum(costo_fijo[i] * y[i] for i in I) + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)

In [ ]:
# Forma alternativa de definir función objetivo: separando costos fijos y transporte
# Opción 1: en líneas separadas
costos_fijos = lp.lpSum(costo_fijo[i] * y[i] for i in I)
costos_transporte = lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
problema += costos_fijos + costos_transporte

# Opción 2: usando variables auxiliares para claridad
z1_fijos = lp.lpSum(f * y_i for f, y_i in zip(costo_fijo.values(), y.values()))
z1_transporte = lp.lpSum(c * x_ij for c, x_ij in zip(costo_transporte.values(), x.values()))
problema += z1_fijos + z1_transporte

# Nota: estas son equivalentes, pero la original es más directa

In [77]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones

____
**Ejemplo**
> La siguiente restricción: $\sum_{i \in I} a_{ij} x_{ij} \geq 1, \; \forall j \in J$ es equivalente a:
>    * `for j in J:`
>        * `model += lp.lpSum(a[i,j]*x[i,j] for i in I) >= 1, 'R1_'+str(j)`
    
**Advertencia**: En `pulp` no es recomendable sobreescribir restricciones, entonces, si ya creaste una restricción y quieres crearla de nuevo para corregir algo, asegúrate de volver a crear el modelo `problema` desde el principio. (Nosotros haremos esto antes de calificar, no te preocupes)

**Pregunta 3 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Explicación detallada — Pregunta 3 (restricción R1)

**Regla de negocio:** cada **CAC** debe tener **exactamente un** depósito que lo atienda (ni cero ni dos).

**Fórmula:** \(\sum_{i \in I} x_{ij} = 1\) para cada `j`.

**El código:**
```text
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, 'R1_' + str(j)
```
1. **`for j in J`** — Creas **una restricción por cada CAC** (cada municipio en la lista `J`).  
2. **`lp.lpSum(x[i, j] for i in I)`** — Suma las `x` de ese CAC `j` con **todos** los depósitos. Si una es 1 y el resto 0, la suma es 1.  
3. **`== 1`** — Obligas asignación **única**.  
4. **`'R1_' + str(j)`** — Nombre de la restricción (útil si PuLP avisa un error; ves qué CAC falló).

**Qué esperas:** tantas filas R1 como CACs tengas en `J`.

---



In [ ]:
# Restricción R1: Cada CAC debe ser atendido por exactamente un depósito
# Para cada CAC j, suma de x[i,j] sobre i == 1
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, 'R1_' + str(j)

In [ ]:
# Forma alternativa de R1: usando enumerate para nombres de restricción
for idx, j in enumerate(J):
    problema += lp.lpSum(x[i, j] for i in I) == 1, f"R1_CAC_{idx}"

# O con comprensión de lista
restricciones_r1 = [lp.lpSum(x[i, j] for i in I) == 1 for j in J]
for restr in restricciones_r1:
    problema += restr

# Verificar número de restricciones agregadas
print(f"Restricciones R1 agregadas: {len(J)}")

**Pregunta 4 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Explicación detallada — Pregunta 4 (restricción R2)

**Reglas de negocio:** (1) no pasarse de la **capacidad** del depósito; (2) no puede salir café de un depósito **cerrado**.

**Fórmula:** \(\sum_j d_j x_{ij} \leq k_i y_i\) para cada depósito `i`.

**El código:**
```text
for i in I:
    problema += lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i], 'R2_' + str(i)
```
1. **`for i in I`** — Una restricción **por cada depósito**.  
2. **`lp.lpSum(produccion[j] * x[i,j] for j in J)`** — Suma la producción de todos los CACs que mandes al depósito `i` (solo cuenta si `x=1`). Es la **carga** del depósito.  
3. **`<= capacidad[i] * y[i]`** — Si `y[i]=1`, el tope es la capacidad. Si `y[i]=0`, el lado derecho es **0**, así que la suma de la izquierda debe ser 0: **nadie** puede estar asignado a un depósito cerrado.

**Qué esperas:** tantas R2 como depósitos en `I`.

**Por qué el producto `capacidad[i] * y[i]`:** une en **una sola línea** “capacidad si está abierto” y “imposible usar si está cerrado”.

---



In [ ]:
# Restricción R2: Capacidad y coherencia con apertura
# Para cada depósito i: suma produccion[j] * x[i,j] <= capacidad[i] * y[i]
for i in I:
    problema += lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i], 'R2_' + str(i)

In [ ]:
# Forma alternativa de R2: calculando carga por depósito de manera explícita
for i in I:
    carga_total = 0
    for j in J:
        carga_total += produccion[j] * x[i, j]
    problema += carga_total <= capacidad[i] * y[i], f"R2_Dep_{i}"

# O usando sum() en lugar de lpSum (menos eficiente pero más legible)
for i in I:
    problema += sum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i], f"R2_alt_{i}"

# Nota: lpSum es preferido porque crea expresiones simbólicas eficientes

In [82]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [83]:
# Resolver el problema de optimización
# problema.solve() ejecuta el solver y devuelve un código de estado
# lp.LpStatus[...] traduce el código a texto (ej: "Optimal")
print(lp.LpStatus[problema.solve()])

Optimal


### `problema.solve()` y `LpStatus`: qué pasó al resolver

**Línea:** `print(lp.LpStatus[problema.solve()])`

**Por partes:**
1. **`problema.solve()`** — Manda a calcular valores de `x` e `y` que cumplan restricciones y optimicen el objetivo. Es el paso “pesado”.
2. **Valor devuelto** — Un código numérico de éxito o fallo; `LpStatus[...]` lo traduce a texto, por ejemplo `Optimal` (óptimo).
3. **`print(...)`** — Solo para **ver en pantalla** si encontró solución.

**Qué esperas:** `Optimal` en un caso bien planteado. Si sale otra cosa, faltan datos, el modelo es infactible o hay error en restricciones.

**Nota:** Hasta que no corre `solve()`, `lp.value(...)` y `.value()` de las variables no tienen sentido o pueden fallar.

---



## Reporte de resultados - Minimización de costos ($z_1$)
---

**Función objetivo $z_1$**

In [ ]:
# Calcular valor de la función objetivo z1 (costo total)
# lp.value(problema.objective): obtiene el valor numérico del objetivo después de solve()
z1 = lp.value(problema.objective)
min_costo = z1  # Guardar para comparaciones
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1/min_costo*100: .2f}%")

Costo Total: $ 4318336.74
Costo Total Relativo al Mínimo Costo:  100.00%


**Función objetivo $z_2$**

**Pregunta 5 (5 puntos)**

* Guarda en una variable `z2` el valor de la satisfacción total dado por la expresión:
> $z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Explicación detallada — Pregunta 5 (medir `z2` después de minimizar costo)

**Contexto:** el **objetivo** del modelo fue **minimizar dinero**, no maximizar cercanía. Esta celda **solo mide** qué tan “contentos” quedaron los CACs según el radio `r`.

**Fórmula de satisfacción:** sumar `produccion[j]` solo cuando el enlace `(i,j)` usado tenga distancia **≤ r** km.

**El código:**
```text
z2 = lp.value(lp.lpSum(
    produccion[j] * x[i, j]
    for j in J
    for i in I
    if distancia[(i, j)] <= r
))
```
1. **`lp.value(...)`** — **Después** de `solve()`, convierte la expresión en un **número** usando los valores 0/1 que quedaron en `x`.  
2. **`for j in J for i in I`** — Recorre todas las combinaciones.  
3. **`if distancia[(i, j)] <= r`** — **Filtro:** solo entran rutas **cercanas** al criterio del enunciado.  
4. **`produccion[j] * x[i,j]`** — Si esa ruta es la elegida (`x=1`) y pasa el filtro, sumas la producción de ese CAC.

**Qué esperas:** un `z2` entre 0 y la producción total. Si muchos CACs quedan lejos del depósito asignado, `z2` será **bajo** aunque el costo sea bajo.

---



In [ ]:
# Calcular z2: satisfacción (producción atendida dentro de r=125 km)
# Suma produccion[j] * x[i,j] solo si distancia[(i,j)] <= r
z2 = lp.value(lp.lpSum(produccion[j] * x[i, j] for j in J for i in I if distancia[(i, j)] <= r))

In [ ]:
# Forma alternativa de verificar estado: con manejo de errores
try:
    status = problema.solve()
    status_text = lp.LpStatus[status]
    print(f"Estado de la solución: {status_text}")
    if status == 1:  # Optimal
        print("Solución óptima encontrada")
    elif status == -1:  # Infeasible
        print("Problema infactible - revisar restricciones")
    else:
        print(f"Estado desconocido: {status}")
except Exception as e:
    print(f"Error al resolver: {e}")

In [ ]:
# Forma alternativa de calcular z1: manualmente sumando valores
z1_manual = 0
for i in I:
    if y[i].value() == 1:
        z1_manual += costo_fijo[i]
for i in I:
    for j in J:
        if x[i, j].value() == 1:
            z1_manual += costo_transporte[i, j]

print(f"z1 con lp.value(): ${lp.value(problema.objective):.2f}")
print(f"z1 manual: ${z1_manual:.2f}")
print(f"Diferencia: ${abs(lp.value(problema.objective) - z1_manual):.6f}")

**Depósitos en operación**

In [ ]:
# Contar depósitos operados
# sum(y[i].value() for i in I): cuenta cuántos y[i] == 1
print("Se decidió operar", sum(y[i].value() for i in I), "depósitos")

Se decidió operar 15.0 depósitos


**Asignación de CACs a Depósitos**

In [ ]:
# Crear tabla de asignación: filas CACs, columnas depósitos abiertos
# 'X' si asignado, '-' si no, 'Error' si valor inesperado
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:  # Solo depósitos operados
            if x[i,j].value() == 1:
                row.append('X')
            elif x[i,j].value() == 0:
                row.append('-')
            else:
                row.append('Error')
    matrix.append(row)

# DataFrame para mostrar tabla: index=J (CACs), columns=depósitos abiertos
df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Santana, Huila","Neiva, Huila","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Filandia, Quindío","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,X,-,-
"Miraflores, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,-,X,-,-,-,-,-,-,-,-,-,-,-,-


### Tabla de asignación (letras `X` y `-`)

**Ideas:**
- Recorre cada **CAC** (`for j in J`). Para cada uno arma una **fila**.
- Solo mira depósitos con **`y[i].value() == 1`** (abiertos).
- Si **`x[i,j].value() == 1`**, escribe `'X'`: ese CAC está asignado a ese depósito.
- Si es 0, escribe `'-'`: ese depósito abierto no atiende a ese CAC.

**El `DataFrame`** arma una tabla legible: filas = CACs, columnas = depósitos **en operación**.

**Para qué sirve:** ver de un vistazo la configuración después del solver, sin leer miles de números 0/1.

---



### Visualizaciones
---

**Mapa de la asignación**

In [ ]:
# Crear mapa con Folium para visualizar asignaciones
# Importar librerías necesarias
import folium
from folium.plugins import BeautifyIcon

# Crear mapa centrado en Colombia
m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

# Agregar marcadores para CACs (círculos azules)
for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)

# Agregar marcadores para depósitos operados (triángulos rojos)
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:  # Solo si abierto
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

# Dibujar líneas entre depósitos y CACs asignados
red = [(i, j) for i in I for j in J if x[i, j].value() > 0]  # Pares asignados
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)

# Mostrar mapa
m

### Mapa (Folium): qué dibuja el código

**Folium** crea mapas web dentro del notebook.

**Marcadores azules (círculos):** cada **CAC** (`cacs_lat_lon`).

**Marcadores rojos (triángulo):** depósitos con **`y[i].value()` > 0**, es decir **abiertos**.

**Líneas negras:** cada par `(i,j)` donde **`x[i,j]` vale 1** en la solución: enlace “este CAC va a este depósito”.

**Centro y zoom:** `[6.2, -74.5]`, zoom 6 — vista general de Colombia.

**En palabras:** mismo resultado que la tabla, pero en geografía.

---



## Modelado - Maximización de satisfacción ($z_2$)
---
A continuación queremos explorar el cambio en las funciones objetivo $z_1$ y $z_2$ cuando se prioriza $z_2$. Las restricciones y variables del problema permanecen igual, pero la solución cambiará.

### Declaración del modelo

In [90]:
problema = lp.LpProblem(sense=lp.LpMaximize)

### Segundo bloque: **maximizar** (`LpMaximize`)

**Código:** `problema = lp.LpProblem(sense=lp.LpMaximize)`

**Diferencia con el primero:** aquí el **sentido** del objetivo es **maximizar** (satisfacción `z2`), no minimizar costo.

**Importante:** al crear un **nuevo** `problema`, hay que **volver a declarar** `x`, `y`, objetivo y restricciones (R1, R2). No se “arrastran” solos desde el modelo anterior.

**Resumen oral:** *“Mismo negocio, mismas reglas, pero ahora el criterio es subir la cobertura cercana, no bajar el costo.”*

---



### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [91]:
x = lp.LpVariable.dicts('atender', [(i,j) for i in I for j in J], cat=lp.LpBinary)
y = lp.LpVariable.dicts('operar', I, cat=lp.LpBinary)

### Función Objetivo
Maximizar satisfacción de los CACs
>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}.$

Esta expresión es equivalente a:
>$\max z_2 = \sum_{j \in J} \sum_{\{i \in I | h_{ij} \leq r\}}d_{j} x_{ij}.$

**Pregunta 6 (5 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Explicación detallada — Pregunta 6 (objetivo `z2`, segundo modelo)

**Cambio de historia:** ahora el optimizador **maximiza** la misma idea de “producción bien atendida cerca”, no el costo.

**El código:**
```text
problema += lp.lpSum(
    produccion[j] * x[i, j]
    for j in J
    for i in I
    if distancia[(i, j)] <= r
)
```
- Igual que la suma de P5, pero aquí va como **función objetivo** de un `problema` declarado con `LpMaximize` arriba.

**Qué espera el solver:** asignaciones que suban esa suma, respetando R1 y R2 (las añades después).

**Trade-off esperado:** `z2` puede subir, pero **el costo** `z1` a menudo **sube**; por eso más adelante calculas `z1` aparte.

---



In [92]:
problema += lp.lpSum(produccion[j] * x[i, j] for j in J for i in I if distancia[(i, j)] <= r)


In [ ]:
# Forma alternativa de contar depósitos operados: con list comprehension
depositos_abiertos = [i for i in I if y[i].value() == 1]
print(f"Depósitos operados: {len(depositos_abiertos)}")
print("Lista:", depositos_abiertos)

# O con filter
depositos_abiertos_filter = list(filter(lambda i: y[i].value() == 1, I))
print("Con filter:", len(depositos_abiertos_filter))

In [94]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones
____

**Pregunta 7 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Explicación — Pregunta 7 (R1 otra vez)

Es **la misma lógica** que P3: un depósito por CAC.

**Por qué repetir:** construyes un **nuevo** objeto `problema` (maximizar `z2`). Las restricciones no se “heredan” solas; hay que volver a añadirlas.

**Código:** mismo patrón `for j in J:` + suma de `x[i,j] == 1` con nombre `R1_...`.

---



In [95]:
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, 'R1_' + str(j)


In [ ]:
# Forma alternativa de crear tabla: usando pandas directamente
asignaciones = {}
for j in J:
    for i in I:
        if y[i].value() == 1:
            asignaciones[(j, i)] = 'X' if x[i, j].value() == 1 else '-'

df_alt = pd.DataFrame(index=J, columns=[i for i in I if y[i].value() == 1])
for j in J:
    for i in df_alt.columns:
        df_alt.loc[j, i] = 'X' if x[i, j].value() == 1 else '-'

print("Tabla alternativa:")
print(df_alt.head())

**Pregunta 8 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Explicación — Pregunta 8 (R2 otra vez)

Igual que P4: capacidad + “solo si el depósito está abierto”.

**Por qué repetir:** mismo motivo que P7: es un modelo nuevo desde cero con las mismas reglas de negocio.

---



In [97]:
for i in I:
    problema += lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i], 'R2_' + str(i)


In [98]:
# Esta celda esta reservada para uso del equipo docente

In [99]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [100]:
print(lp.LpStatus[problema.solve()])

Optimal


## Reporte de resultados - Maximización de satisfacción ($z_2$)
---

**Función objetivo $z_1$**

**Pregunta 9 (5 puntos)**

* Guarda en una variable `z1` el valor del costo total de operación y transporte:
>`# Para desarrollo del estudiante`

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Explicación detallada — Pregunta 9 (`z1` en la solución que maximiza `z2`)

**Situación:** el solver **maximizó** `z2`. El valor del costo **no** sale del `.objective` de este modelo (ese es `z2`). Tú quieres el **dinero total**.

**El código** vuelve a armar la suma de **fijos + transporte** y la envuelve en `lp.value`:
```text
z1 = lp.value(
    lp.lpSum(costo_fijo[i] * y[i] for i in I)
    + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
)
```

**En palabras:** “Con la solución ya encontrada, cuánto me costaría en realidad operar y transportar así.”

**Para qué sirve:** comparar con `min_costo` del primer modelo y ver **cuánto más caro** es priorizar cercanía.

---



In [101]:
z1 = lp.value(lp.lpSum(costo_fijo[i] * y[i] for i in I) + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J))


In [102]:
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1 / min_costo * 100: .2f}%")

Costo Total: $ 5254425.06
Costo Total Relativo al Mínimo Costo:  121.68%


In [103]:
# Esta celda esta reservada para uso del equipo docente

**Función objetivo $z_2$**

In [104]:
z2 = lp.value(problema.objective)
print(f"Satisfacción Total: {z2: .2f}")
print(
    f"Satisfacción Total Relativa al Total de Producción: {z2 / sum(produccion.values()) * 100: .2f}%"
)

Satisfacción Total:  398.33
Satisfacción Total Relativa al Total de Producción:  94.79%


**Depósitos en operación**

In [105]:
print("Se decidió operar", sum([y[i].value() for i in I]), "depósitos")

Se decidió operar 19.0 depósitos


**Asignación de CACs a Depósitos**

In [106]:
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i, j].value() == 1:
                row.append("X")
            elif x[i, j].value() == 0:
                row.append("-")
            else:
                row.append("Error")
    matrix.append(row)

df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Bogotá, Cundinamarca","Santana, Huila","Neiva, Huila","Santa Marta, Magdalena","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Calarcá, Quindío","Filandia, Quindío","Pereira, Risaralda","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,-,-,-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Miraflores, Boyacá",-,-,-,-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


### Visualizaciones
---

**Mapa de la asignación**

In [107]:
m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)

m

## Modelado - Maximización de satisfacción ($z_2$) con restricción de costos ($z_1$)
---
Por último, queremos encontrar un solución intermedia entre la que minimiza los costos y la que maximiza la satisfacción. Hay varias maneras de hacer esto. La que vamos a implementar es colocar una restricción sobre los costos $z_1$ que esté entre los valores obtenidos en los dos casos anteriores.

Recordemos que al minimizar los costos, se obtuvo un costo total de 4,318,336.74. Este costo será nuestro punto de referencia. No podemos obtener un costo menor a este. Por otro lado, al maximizar la satisfacción, obtuvimos un costo de 4,837,569.38. Así que en el peor de los casos, el costo es aproximadamente 12.02% mayor al primer caso. Entonces, para obtener una solución intermedia, debemos escoger un umbral entre estos dos valores para crear una restricción sobre los costos. Una posibilidad es restringir que el costo total $z_1$ sea a lo sumo 2% mayor que el mejor costo mientras se maximiza la satisfacción $z_2$.

### Tercer bloque: compromiso costo–servicio

El texto arriba resume la idea: se **maximiza** otra vez `z2`, pero se añade **R3**: el costo total no puede superar el **102 %** del mejor costo conocido (`z1_`).

**En palabras:** no buscas ni lo más barato ni lo más caro; buscas **mucho servicio** sin alejarte más de un 2 % del mínimo de gasto.

Las variables `x`, `y` y las restricciones R1 y R2 son las mismas **ideas** que antes; lo nuevo es el **tope de dinero** (R3).

---



### Declaración del modelo

In [ ]:
# Forma alternativa de crear mapa: sin BeautifyIcon, con marcadores simples
import folium

m_alt = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

# CACs con círculos simples
for j, lat_lon in cacs_lat_lon.items():
    folium.CircleMarker(
        location=lat_lon,
        radius=5,
        color='blue',
        fill=True,
        popup=j
    ).add_to(m_alt)

# Depósitos con triángulos simples
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            icon=folium.Icon(color='red', icon='info-sign'),
            popup=i
        ).add_to(m_alt)

# Líneas
red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]],
        color="green",
        weight=2
    ).add_to(m_alt)

# Mostrar mapa alternativo
m_alt

### (Recordatorio) Nuevo modelo = volver a armar todo

Aquí otra vez **`LpMaximize`**: es el **tercer** problema (compromiso con R3). La idea es la misma que en el segundo bloque: nuevas `x`, `y`, nuevo objetivo y nuevas copias de R1 y R2, más R3.

---



### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{Si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [109]:
x = lp.LpVariable.dicts("atender", [(i, j) for i in I for j in J], cat=lp.LpBinary)
y = lp.LpVariable.dicts("operar", I, cat=lp.LpBinary)

### Función Objetivo
Maximizar satisfacción de los CACs
>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

**Pregunta 10 (5 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Explicación — Pregunta 10 (objetivo `z2`, tercer modelo)

Otra vez maximizas la misma suma ponderada por cercanía que en P6.

**Diferencia con el modelo 2:** después agregarás **R3**, un tope explícito al costo. El objetivo sigue siendo “subir `z2`”, pero **dentro** de un corral de dinero.

---



In [ ]:
# Forma alternativa de calcular z2: con bucles explícitos
z2_alt = 0
for j in J:
    for i in I:
        if distancia[(i, j)] <= r and x[i, j].value() == 1:
            z2_alt += produccion[j]

print(f"z2 original: {z2}")
print(f"z2 alternativa: {z2_alt}")
print(f"Iguales: {z2 == z2_alt}")

# O usando sum con condición
z2_sum = sum(produccion[j] for j in J for i in I if distancia[(i, j)] <= r and x[i, j].value() == 1)
print(f"z2 con sum: {z2_sum}")

In [111]:
# Esta celda esta reservada para uso del equipo docente

In [112]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones
____

**Pregunta 11 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Explicación — Pregunta 11 (R1, tercer modelo)

Misma restricción de asignación única. Sin R1 el modelo podría dejar CACs sin depósito o con varios.

---



In [ ]:
# Forma alternativa de maximizar z2 con restricción de costo: usando presupuesto fijo
presupuesto_max = 2000000  # Ejemplo: 2M de pesos
problema_alt = lp.LpProblem(sense=lp.LpMaximize)

# Variables (reusar x, y o crear nuevas)
x_alt = lp.LpVariable.dicts("atender_alt", [(i, j) for i in I for j in J], cat=lp.LpBinary)
y_alt = lp.LpVariable.dicts("operar_alt", I, cat=lp.LpBinary)

# Restricciones (iguales)
for j in J:
    problema_alt += lp.lpSum(x_alt[i, j] for i in I) == 1
for i in I:
    problema_alt += lp.lpSum(produccion[j] * x_alt[i, j] for j in J) <= capacidad[i] * y_alt[i]

# Nueva restricción: costo <= presupuesto
problema_alt += lp.lpSum(costo_fijo[i] * y_alt[i] for i in I) + lp.lpSum(costo_transporte[i, j] * x_alt[i, j] for i in I for j in J) <= presupuesto_max

# Objetivo: max z2
problema_alt += lp.lpSum(produccion[j] * x_alt[i, j] for j in J for i in I if distancia[(i, j)] <= r)

# Resolver
problema_alt.solve()
print(f"Estado: {lp.LpStatus[problema_alt.solve()]}")
print(f"z2 máxima con presupuesto {presupuesto_max}: {lp.value(problema_alt.objective)}")

In [114]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 12 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Explicación — Pregunta 12 (R2, tercer modelo)

Misma restricción de capacidad y coherencia con `y[i]`.

---



In [115]:
for i in I:
    problema += lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i], 'R2_' + str(i)


In [116]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 13 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R3'` y añádela al modelo:
>El costo total no debe superar en más de un 2% al mejor costo obtenido.
>>`# Para desarrollo del estudiante`

**Nota:** Utiliza el valor `z1_` a continuación como el mejor costo obtenido. Inclúyelo en la restricción según sea conveniente.

In [117]:
z1_ = 4318336.74

### Explicación detallada — Pregunta 13 (restricción R3, tope de costo)

**Idea:** “Sí quiero mucha satisfacción (`z2`), pero **no** me aparto más de un 2 % del **mejor costo** conocido (`z1_`).”

**El código:**
```text
problema += (
    lp.lpSum(costo_fijo[i] * y[i] for i in I)
    + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
    <= z1_ * 1.02
), 'R3'
```
1. **Primera suma** — Costos fijos según depósitos abiertos.  
2. **Segunda suma** — Costos de transporte según `x`.  
3. **`<= z1_ * 1.02`** — Límite: como mucho **102 %** del mejor `z1` del primer escenario (el valor `z1_` está fijado en la celda anterior).  
4. **`'R3'`** — Nombre de la restricción (en el enunciado te lo piden así).

**Tecnicismo:** `z1_` es un **número fijo** (dato), no una variable; por eso va al lado derecho sin `lp.value`.

---



In [118]:
problema += lp.lpSum(costo_fijo[i] * y[i] for i in I) + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J) <= z1_ * 1.02, 'R3'


In [119]:
# Esta celda esta reservada para uso del equipo docente

In [120]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [121]:
print(lp.LpStatus[problema.solve()])

Optimal


## Reporte de resultados - Maximización de satisfacción ($z_2$) con restricción de costos ($z_1$)
---

**Función objetivo $z_1$**

**Pregunta 14 (5 puntos)**

* Guarda en una variable `z1` el valor del costo total de operación y transporte:
>`# Para desarrollo del estudiante`

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Explicación detallada — Pregunta 14 (`z1` final del tercer modelo)

Igual filosofía que P9: el objetivo optimizado era `z2` con R1, R2 y **R3**. Para saber el **costo** de esa decisión, vuelves a evaluar la suma fijos + transporte con `lp.value`.

**Para qué sirve:** verificar que el costo respeta el tope (debería ser ≤ `z1_*1.02`) y comparar con `min_costo` y con el caso “solo maximizar `z2`”.

---



In [ ]:
# Forma alternativa de tabla para segundo modelo: con colores
matrix_alt = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i,j].value() == 1:
                if distancia[(i, j)] <= r:
                    row.append('X (cerca)')  # Verde en mente
                else:
                    row.append('X (lejos)')  # Rojo en mente
            else:
                row.append('-')
    matrix_alt.append(row)

df_alt2 = pd.DataFrame(matrix_alt, index=J, columns=[i for i in I if y[i].value() == 1])
print("Tabla con distancias:")
print(df_alt2.head(10))

In [123]:
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1 / min_costo * 100: .2f}%")

Costo Total: $ 4403765.23
Costo Total Relativo al Mínimo Costo:  101.98%


In [124]:
# Esta celda esta reservada para uso del equipo docente

**Función objetivo $z_2$**

In [125]:
z2 = lp.value(problema.objective)
print(f"Satisfacción Total: {z2: .2f}")
print(f"Satisfacción Total Relativa al Total de Producción: {z2 / sum(produccion.values()) * 100: .2f}%")

Satisfacción Total:  396.67
Satisfacción Total Relativa al Total de Producción:  94.39%


**Depósitos en operación**

In [126]:
print("Se decidió operar", sum(y[i].value() for i in I), "depósitos")

Se decidió operar 17.0 depósitos


**Asignación de CACs a Depósitos**

In [127]:
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i, j].value() == 1:
                row.append("X")
            elif x[i, j].value() == 0:
                row.append("-")
            else:
                row.append("Error")
    matrix.append(row)

df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Santana, Huila","Neiva, Huila","Santa Marta, Magdalena","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Calarcá, Quindío","Filandia, Quindío","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Miraflores, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


### Visualizaciones
---

**Mapa de la asignación**

In [128]:
m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)

for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)
m

### Reflexión
---
¿De qué forma podrías obtener soluciones intermedias adicionales? ¿Podrías presentarlas gráficamente como una frontera de Pareto? ¿Si tuvieras que recomendar alguna solución, con qué criterio la escogerías?

In [ ]:
# Forma alternativa de mapa para segundo modelo: diferenciando cercanos y lejanos
m_alt2 = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

# CACs
for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(location=lat_lon, tooltip=j, icon=folium.Icon(color='blue')).add_to(m_alt2)

# Depósitos
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(location=lat_lon, tooltip=i, icon=folium.Icon(color='red')).add_to(m_alt2)

# Líneas: verdes para cercanas, rojas para lejanas
for i in I:
    for j in J:
        if x[i, j].value() > 0:
            color = 'green' if distancia[(i, j)] <= r else 'red'
            folium.PolyLine(
                [depositos_lat_lon[i], cacs_lat_lon[j]],
                color=color,
                weight=2,
                opacity=0.7
            ).add_to(m_alt2)

# Leyenda
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 150px; height: 90px; 
     background-color: white; border:2px solid grey; z-index:9999; font-size:14px;">
     &nbsp; <b>Leyenda</b> <br>
     &nbsp; Verde: ≤ 125 km <br>
     &nbsp; Rojo: > 125 km <br>
</div>
'''
m_alt2.get_root().html.add_child(folium.Element(legend_html))

m_alt2

## Respuesta guiada (recuadros)

<div style="border:2px solid #1565c0; border-radius:12px; padding:16px 20px; margin:16px 0; background:#e3f2fd;">
<strong style="color:#0d47a1;">Recuadro 1 — ¿Cómo obtener más soluciones intermedias?</strong>

<p style="margin:10px 0 0 0; line-height:1.55;">La idea del tercer modelo ya es una <em>solución intermedia</em>: maximizar la satisfacción <code>z2</code> sin pasarse de un tope de costo (por ejemplo el costo mínimo <code>z1*</code> más un 2 %).</p>

<p style="margin:10px 0 0 0; line-height:1.55;"><strong>Para obtener más puntos intermedios</strong> puedes repetir ese mismo tipo de problema varias veces cambiando solo el tope: resolver <em>maximizar</em> <code>z2</code> sujeto a <code>z1 ≤ (1 + ε) · z1*</code> con distintos <code>ε</code> (1 %, 2 %, 5 %, 10 %…). Cada <code>ε</code> da otra configuración de depósitos y asignaciones. También puedes hacer el camino inverso: <em>minimizar</em> <code>z1</code> exigiendo que <code>z2</code> no baje de cierto valor (distintos pisos de satisfacción).</p>
</div>

<div style="border:2px solid #2e7d32; border-radius:12px; padding:16px 20px; margin:16px 0; background:#e8f5e9;">
<strong style="color:#1b5e20;">Recuadro 2 — ¿Frontera de Pareto en un gráfico?</strong>

<p style="margin:10px 0 0 0; line-height:1.55;"><strong>Sí.</strong> Tienes dos objetivos en tensión: <strong>menor costo</strong> <code>z1</code> y <strong>mayor satisfacción</strong> <code>z2</code> (producción bien atendida dentro del radio <code>r</code> km). Un punto es <strong>Pareto-eficiente</strong> si no puedes mejorar uno sin empeorar el otro.</p>

<p style="margin:10px 0 0 0; line-height:1.55;"><strong>Cómo graficarlo:</strong> en el eje horizontal el costo <code>z1</code> (o el porcentaje sobre el mínimo), y en el vertical la satisfacción <code>z2</code> (o el porcentaje sobre la producción total). Cada corrida con un <code>ε</code> distinto (o un piso distinto de <code>z2</code>) genera un par <code>(z1, z2)</code>. Al marcar esos puntos obtienes una <strong>aproximación a la frontera de Pareto</strong>.</p>

<p style="margin:10px 0 0 0; line-height:1.55;"><em>Nota práctica:</em> en Python puedes usar <code>matplotlib</code> (<code>scatter</code> o <code>plot</code>) con listas de valores guardados en un bucle.</p>
</div>

<div style="border:2px solid #e65100; border-radius:12px; padding:16px 20px; margin:16px 0; background:#fff3e0;">
<strong style="color:#bf360c;">Recuadro 3 — ¿Qué solución recomendar y con qué criterio?</strong>

<p style="margin:10px 0 0 0; line-height:1.55;">Depende del <strong>decisor</strong> y de <strong>cuánto acepta pagar</strong> por ganar servicio. Ejemplos de criterio:</p>

<ul style="margin:8px 0 0 0; line-height:1.55;">
<li><strong>Tope de presupuesto:</strong> no más de X % sobre el costo mínimo; entonces eliges sobre la frontera el punto con mayor <code>z2</code> que cumpla ese tope (como el +2 % del laboratorio).</li>
<li><strong>Regla del codo:</strong> donde subir un poco más <code>z2</code> exige disparar mucho <code>z1</code>, suele ser zona sensata para parar.</li>
<li><strong>Regla explícita:</strong> cada millón extra debe compensar al menos con Y unidades de satisfacción.</li>
</ul>

<p style="margin:10px 0 0 0; line-height:1.55;"><strong>Respuesta defendible:</strong> mostrar la frontera al decisor y recomendar el punto que respeta un <strong>límite de costo claro</strong>, o el que maximiza <code>z2</code> bajo ese límite.</p>
</div>

